# ORAX-KG — Pipeline Notebook

End-to-end pipeline.

**Stages:**
1. Ontology-guided triple extraction (LLM)
2. Triple embedding & alignment
3. Consensus clustering
4. LLM cluster validation → Ontology expansion with novel relations


---
## Step 1 — Ontology-Guided Triple Extraction

Loads the relation schema, splits it into known/hidden sets, then runs the LLM on each sentence to extract `(subject, relation, object)` triples.

**Output:** `results/notebook_run/02_extraction/extractions.jsonl

In [13]:
! python scripts/run_extraction.py --data data/raw_data/DATASET_EXAMPLE.json --schema data/ontologies/ReTACRED_ontology.json --output results/my_run --base-url https://unflippant-concetta-bilgiest.ngrok-free.dev/v1 --model Qwen/Qwen3-14B

ORAX-KG Relation Extraction

  Output directory: results\my_run

  Connecting to vLLM server at https://unflippant-concetta-bilgiest.ngrok-free.dev/v1...
  Model: Qwen/Qwen3-14B

  Loading data from data/raw_data/DATASET_EXAMPLE.json...
   Loaded 13 triples

  Loading schema from data/ontologies/ReTACRED_ontology.json...
   Loaded schema from data/ontologies/ReTACRED_ontology.json
   Domain types: 2
   Total relation types: 120
   Loaded 120 ontology entries
   Found 24 relations with multiple type signatures

  Splitting ontology...

 Ontology Split:
   Total ontology entries: 120
   Known:  32 relations -> 96 entries
   Hidden: 8 relations -> 24 entries

  Saving ontology splits...
   Saved to results\my_run\01_ontology

  Sampling test data...
Sampling Statistics:
   Known samples:  9
   Hidden samples: 4
   Total sampled:  13

  Starting fresh extraction

  Extracting from index 0...
   Total samples: 13 | Remaining: 13
--------------------------------------------------------------

---
## Step 2 — Triple Embedding & Alignment

Embeds extracted triples and known ontology patterns into four independent views, then aligns each triple against the ontology using cosine similarity.

**Output:** `results/notebook_run/03_alignment/`

In [8]:
! python scripts/run_alignment.py --run-dir results/my_run --output results/my_run --embedder sentence-transformers/all-MiniLM-L6-v2 --threshold 0.90 --device cpu

ORAX-KG Ontology Alignment

  Loading from:     results\my_run
  Output directory: results\my_run\03_alignment

  Loading extraction artifacts...
   Loaded 13 extractions
   Loaded known ontology (96 entries)

  Preparing extracted triples...

  Initializing embedder: sentence-transformers/all-MiniLM-L6-v2
  Model loaded on CPU with float32

  Embedding extracted triples...

  Embedding ontology patterns...

  Saving embeddings...
   Saved to results\my_run\03_alignment

  Computing similarities...

  Aligning triples (threshold=0.9)...

  Alignment complete!
   Aligned:          4
   Novel candidates: 9
   Artifacts saved:  results\my_run\03_alignment


`torch_dtype` is deprecated! Use `dtype` instead!


---
## Step 3 — Consensus Clustering

Groups unaligned triples into semantically coherent clusters using a multi-algorithm ensemble (Spectral, HDBSCAN, Leiden).

**Output:** `results/notebook_run/04_clustering/`

In [9]:
! python scripts/run_clustering.py --run-dir results/my_run --similarity-threshold 0.60 --consensus-runs 3

ORAX-KG Consensus Clustering

  Loading from: results\my_run

  Loading artifacts...
   ✓ Loaded 13 extractions, 13 alignment results, 13 embeddings

✓ Built ground truth for 13 extracted triples

  Computing inter-triple similarities...

  Running consensus clustering...
 Initialized with algorithms: ['spectral', 'hdbscan', 'leiden']
   Type stratification: 5 type-pair groups
   Clustering person:person (4 items)
   Clustering person:title (2 items)
   Preserving 4 singletons as potential novel relations
→ Mode 'relation' produced 6 clusters

  Evaluating clustering...

 NOVEL DISCOVERY EVALUATION
   Samples: 2 | GT classes: 2 | Pred clusters: 2
   B³ F1: 1.000 | V-measure: 1.000 | ARI: 1.000

 ROBUSTNESS CHECK (Known Relations)
   Samples: 7 | GT classes: 4 | Pred clusters: 4
   B³ F1: 1.000 | V-measure: 1.000 | ARI: 1.000

 OVERALL PIPELINE PERFORMANCE
   Samples: 9 | GT classes: 6 | Pred clusters: 6
   B³ F1: 1.000 | V-measure: 1.000 | ARI: 1.000

  Saving results

  Clustering com

---
## Step 4 — LLM Cluster Validation

For each cluster, the LLM applies a four-step protocol to decide whether it represents a genuinely novel ontology relation.

**Output:** `results/notebook_run/05_validation/`

In [11]:
! python scripts/run_validation.py --extraction-dir results/my_run --clusters results/my_run/04_clustering/clusters.json --embeddings results/my_run/03_alignment/extracted_embeddings.pt --output results/my_run/05_validation --base-url https://unflippant-concetta-bilgiest.ngrok-free.dev/v1 --model Qwen/Qwen3-32B

ORAX-KG Cluster Validation

  Extraction dir: results\my_run
  Output dir:     results\my_run\05_validation

  Connecting to vLLM server at https://unflippant-concetta-bilgiest.ngrok-free.dev/v1...
  Model: Qwen/Qwen3-32B

  Loading ontology...
   Known relations:  96
   Ontology classes: 16

  Loading clusters from results/my_run/04_clustering/clusters.json...
   Loaded clusters from results/my_run/04_clustering/clusters.json
   Relation-mode clusters: 6

  Loading embeddings from results/my_run/03_alignment/extracted_embeddings.pt...
   Loaded 13 embeddings from results/my_run/03_alignment/extracted_embeddings.pt

  Computing inter-triple similarities...

VALIDATING 6 CLUSTERS

Cluster 0: Skipping (size 3 < min 5)
Cluster 1: Skipping (size 1 < min 5)
Cluster 2: Skipping (size 2 < min 5)
Cluster 3: Skipping (size 1 < min 5)
Cluster 4: Skipping (size 1 < min 5)
Cluster 5: Skipping (size 1 < min 5)

VALIDATION SUMMARY

  Novel Relations Accepted:     0
  Rejected (Existing Match):    0


---
## Full Pipeline

Runs all four stages end-to-end from the config file.

In [ ]:
! python scripts/run_full_pipeline.py --config config.yaml